# TetraFT — QAFT run (0.8B on Kaggle)

**Attach datasets**
- `tetraft-code` — flat `.py` modules
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run (Qwen + recent `transformers` for `qwen3_5`) |
| Flow | inventory → original PPL → shock PPL → QAFT |

**Presets** (tokens @ 1×512×8 = 4096/step)

| Preset | Steps | ≈ tokens | LR |
|--------|------:|---------:|----|
| `short` | 200 | 0.82M | linear→0 |
| `full_smoke` | 1280 | 5.24M | linear→0 (done: PPL~79) |
| **`scale_25m`** | 6104 | **25.0M** | **cosine, floor 0.1×** |
| `scale_50m` | 12207 | 50.0M | cosine, floor 0.1× |

Logic in `run_smoke.py` — notebook is glue only.

In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

In [ ]:
from run_smoke import run_smoke
import argparse

# Phase 1c: scale_25m (~25M tok, cosine LR + 0.1 floor). Optional: scale_50m.
PRESET = "scale_25m"

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir="/kaggle/working/checkpoints",
    seq_length=None,
    batch_size=None,
    max_steps=None,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=False,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    seed=42,
    device_map="auto",
)
results = run_smoke(ns)
keys = ["preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
        "loss_finite", "tokens_seen", "tokens_budget", "steps_ran"]
print({k: results[k] for k in keys if k in results})
if "ppl_after_smoke" in results and "ppl_original" in results:
    print("after/orig =", results["ppl_after_smoke"] / results["ppl_original"])

Artifacts: `/kaggle/working/checkpoints/{linear_inventory,smoke_results,checkpoint-*}`

**Baselines so far**
- orig ≈ 17.7
- shock ≫ 1e6
- 5.2M tok → PPL ≈ 79.4 (gap ≈ 4.5×)

**Phase 1c success:** end PPL **&lt; 79**, still improving after λ=1; LR should not hit 0 (cosine floor).

Then Phase 2 ablations on fixed data.